In [0]:
import dlt
from pyspark.sql.functions import col, count, countDistinct, max, min, datediff, hour, sum, when


@dlt.table(
    name="silver_unified_logs",
    comment="Bảng gộp Batch và Streaming, đã xóa trùng lặp và chuẩn hóa thời gian"
)
def silver_unified():
    df_history = dlt.read("bronze_music_data")
    df_live = dlt.read("live_music_logs")
    
    df_combined = df_history.unionByName(df_live, allowMissingColumns=True)
    
    df_clean = df_combined.withColumn("timestamp", col("timestamp").cast("timestamp")) \
        .filter(
            col("user_id").isNotNull() & 
            col("timestamp").isNotNull() & 
            col("recording_msid").isNotNull()
        )
        
    df_final_silver = df_clean.dropDuplicates(["user_id", "recording_msid", "timestamp"])
    
    return df_final_silver



@dlt.table(
    name="gold_user_features",
    comment="Bảng chứa Features phục vụ mô hình học máy dự đoán Churn (Đã nâng cấp)"
)
def gold_features():
    df = dlt.read("silver_unified_logs")
    
    df = df.withColumn("is_night", when((hour("timestamp") >= 22) | (hour("timestamp") <= 5), 1).otherwise(0))
    
    df_grouped = df.groupBy("user_id").agg(
        count("recording_msid").alias("total_listens"),
        countDistinct("artist_name").alias("unique_artists"),
        countDistinct("track_name").alias("unique_tracks"), # Feature mới từ dữ liệu bạn vừa bổ sung!
        sum("is_night").alias("night_listens"),
        max("timestamp").alias("last_listen_date"),
        min("timestamp").alias("first_listen_date")
    )
    
    df_features = df_grouped \
        .withColumn("tenure_days", datediff(col("last_listen_date"), col("first_listen_date")) + 1) \
        .withColumn("daily_listen_rate", col("total_listens") / col("tenure_days")) \
        .withColumn("night_listen_ratio", col("night_listens") / col("total_listens")) \
        .withColumn("artist_diversity", col("unique_artists") / col("total_listens")) \
        .withColumn("track_diversity", col("unique_tracks") / col("total_listens")) # Tỷ lệ đa dạng bài hát
        
    df_gold_final = df_features.select(
        "user_id", 
        "total_listens", 
        "daily_listen_rate", 
        "night_listen_ratio", 
        "artist_diversity",
        "track_diversity",
        "tenure_days"
    )
    
    return df_gold_final